# Notebook 8: Calculations and Texts for Streamlit 

In [ ]:
#load libraries 
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import numpy as np
import pandas as pd

In [ ]:
#load dataset
df = pd.read_csv("../../data/final_datasets/df_clusters_milieuschutz.csv")

feat = [c for c in df.columns if c.startswith(("soc_", "re_", "com_"))]
Xs = StandardScaler().fit_transform(df[feat].fillna(df[feat].median()))

des_idx   = df.index[df["ms_over50"] == 1]
watch_idx = df.index[df["on_watchlist"]]

D = cdist(Xs[watch_idx], Xs[des_idx])
df.loc[watch_idx, "nearest_designated"] = df.loc[des_idx[D.argmin(axis=1)], "plr_name"].values

watch = df.loc[watch_idx].copy()
watch["rank"] = watch["oof_prob"].rank(ascending=False, method="min").astype(int)

out = watch.sort_values("rank")[["plr_id", "plr_name", "rank", "oof_prob", "nearest_designated"]]
out.to_csv("watchlist_display.csv", index=False)
print(out.to_string(index=False))

  plr_id               plr_name  rank  oof_prob     nearest_designated
 1401044             Uferstraße     1  0.999109 Westliche Müllerstraße
 1300731          Koloniestraße     2  0.997601           Leopoldplatz
 1100416            Arkonaplatz     3  0.995976      Teutoburger Platz
 7100206     Großgörschenstraße     4  0.992505  Richard-Sorge-Viertel
 1200623            Stephankiez     5  0.986148  Richard-Sorge-Viertel
 1401049            Schulstraße     6  0.985959     Drontheimer Straße
 4501147      Leon-Jessel-Platz     7  0.984600      Bayerischer Platz
 1300836  Humboldthain Nordwest     8  0.984006             Körnerpark
 1200522     Elberfelder Straße     9  0.983263        Feuerbachstraße
 1200628      Lüneburger Straße    10  0.979387      Köpenicker Straße
 9200511      Griechischer Park    11  0.977478             Letteplatz
 1300834          Brunnenstraße    12  0.976187               Rollberg
 2200211           Chamissokiez    13  0.975650            Wrangelkiez
 36011

## Nearest Designated Match Interpretation 

The nearest designated match for each watchlist PLR often lies in a different part of the city (Großgörschenstraße in Schöneberg maps to Richard-Sorge-Viertel in Friedrichshain, for example), which is meaningful because geography is not a model feature. The resemblance is structural, not spatial. 

Several designated areas recur as the closest match for multiple watchlist PLR (Richard-Wagner-Straße three times, Teutoburger Platz and Letteplatz twice each), reflecting that different undesignated areas can share the same designated profile. The four Charlottenburg-Wilmersdorf entries (Leon-Jessel-Platz, Schloßstraße, Amtsgerichtsplatz, Wilhelmsaue) cluster in the lower half of the ranking, consistent with the weaker signal in that region already documented as a data property, not a model limitation. 

The comparison rests on the same features as the resemblance score and uses unweighted Euclidean distance, so it should be read as an illustrative reference area rather than an independent confirmation.

## Texts for Streamlit Profiles 

Undesignated 
-	Not on the watchlist: this area is shown here for context. The watchlist highlights the undesignated neighbourhoods whose profile most closely matches the designated ones.
-	Resemblance, not risk: That match is a similarity to the designated pattern, not a forecast of gentrification or displacement, and it says nothing about individual residents. 
-	Not on the watchlist ≠ safe: The underlying label reflects political designation, not confirmed displacement, and areas change over time; a low match today means "does not currently resemble the designated areas", not "will never qualify".

Designated 
-	Under Milieuschutz protection: This area already carries a designated social-preservation status (§172 Milieuschutz) under Berlin housing policy. 
-	Part of the reference set: Designated areas are the examples the tool learns from — their shared profile defines the pattern that the watchlist looks for elsewhere. 
-	Reference, not prediction: Because this area is already protected, no resemblance assessment is shown for it; it informs the model rather than being scored by it. 
-	Designation is political, not automatic: The status reflects an administrative district-level decision to protect the area, not a measured level of gentrification or displacement — the tool treats it as a given, not as something it evaluates.

Watchlist 

On the watchlist · rank {rank} of 25
Resemblance score {score} · closest designated match: {nearest_designated}

-	On the watchlist: this undesignated area is among the closest matches to Berlin's already-designated Milieuschutz neighbourhoods.
-	Resemblance, not risk: this score captures how closely this area's profile matches the pattern of already designated neighbourhoods. It is not a forecast of gentrification or displacement, and it says nothing about individual residents. 
-	A flag for closer review, not a verdict: A high match means the area may warrant a closer look by the district. A starting point to prepare a preliminary study (vorbereitende Untersuchung), not a decision that anything is happening. 
-	Compared against a political label: the designated areas the score matches against reflect administrative decisions, not confirmed displacement; the watchlist surfaces where that same pattern appears in areas not yet protected.
